# Adaptive MEDAL / AB-MCTS on GSM8K

Select a GPU runtime, then run the cells in order. The notebook loads one LLaDA model, reuses existing runtime/Drive weights, runs the genuine Gaussian policy, and saves search history, posterior updates, NFEs, diversity metrics and tree diagrams to Google Drive.

Defaults: `Lc=20`, `C=3`, `K1=3`, `K2=5`, seed 1, 256 output positions, no initialization cap. Each expansion evaluates only its immediate information gain; no rollout is used. Incomplete searches save diagnostics and skip final decoding.

**Before using a fresh Colab checkout:** push the new AB-MCTS files to `adi`, or copy them into the checkout. Required files are `src/dlm_project/decoding/ab_medal.py`, `scripts/run_ab_medal_single.py`, and `scripts/compare_medal_ab_medal.py`. This local notebook has not been published to GitHub automatically.

CPU fixtures reached three depth-20 candidates at 23 and 61 initialization NFEs. Those synthetic checks do not establish the real-model gate or a GPU speed improvement. This notebook reports the real-model gate only after observing the requested candidates.

## 1. Checkout and dependencies

The setup follows `MEDAL_MCTS_Demo.ipynb`. For the private repository, use the Colab secret `GITHUB_TOKEN` or the hidden token prompt. Existing checkout changes are preserved: setup stops before updating a dirty checkout.

In [ ]:
from pathlib import Path
import getpass
import os
import subprocess
import sys
import tempfile

REPOSITORY = "CooL-Legend/CSCI544-DLM"
BRANCH = "adi"
REPO_URL = f"https://github.com/{REPOSITORY}.git"
IN_COLAB = Path("/content").is_dir()
REPO_DIR = Path("/content/CSCI544-DLM") if IN_COLAB else next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/dlm_project").is_dir()
)

def run_git(args, token=None):
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    with tempfile.TemporaryDirectory() as tmp:
        if token:
            askpass = Path(tmp) / "askpass.py"
            askpass.write_text(
                "#!/usr/bin/env python3\nimport os, sys\n"
                "print('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['COLAB_GH_TOKEN'])\n"
            )
            askpass.chmod(0o700)
            env.update(GIT_ASKPASS=str(askpass), COLAB_GH_TOKEN=token)
        return subprocess.run(["git", "-c", "credential.helper=", *args], env=env, capture_output=True, text=True)

def remote_git(args):
    result = run_git(args)
    if result.returncode:
        from google.colab import userdata
        try:
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
        if not token:
            token = getpass.getpass("GitHub token with read access to this private repository: ")
        try:
            result = run_git(args, token)
        finally:
            del token
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or "GitHub clone/update failed.")

def ensure_checkout():
    def local_git(*args):
        return subprocess.check_output(["git", "-C", str(REPO_DIR), *args], text=True).strip()

    if REPO_DIR.exists():
        if not (REPO_DIR / ".git").is_dir():
            raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout. Choose another REPO_DIR.")
        if local_git("remote", "get-url", "origin") != REPO_URL:
            raise RuntimeError("The existing checkout belongs to a different repository.")
        if local_git("status", "--porcelain"):
            raise RuntimeError("The checkout has local changes. Save them or choose another REPO_DIR before updating.")
        fetch_specs = local_git("config", "--get-all", "remote.origin.fetch").splitlines()
        branch_ref = f"refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"
        if not any(spec.lstrip("+") in (branch_ref, "refs/heads/*:refs/remotes/origin/*") for spec in fetch_specs):
            local_git("remote", "set-branches", "--add", "origin", BRANCH)
        remote_git(["-C", str(REPO_DIR), "fetch", "origin", branch_ref])
        exists = subprocess.run(["git", "-C", str(REPO_DIR), "show-ref", "--verify", "--quiet", f"refs/heads/{BRANCH}"]).returncode == 0
        if exists:
            local_git("switch", BRANCH)
        else:
            local_git("switch", "--create", BRANCH, "--track", f"origin/{BRANCH}")
        local_git("merge", "--ff-only", f"origin/{BRANCH}")
    else:
        remote_git(["clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
    if local_git("branch", "--show-current") != BRANCH or local_git("rev-parse", "HEAD") != local_git("rev-parse", f"origin/{BRANCH}"):
        raise RuntimeError("The checkout differs from origin/adi. Review local commits before running experiments.")
    print(f"Code ready: {REPO_DIR} | branch: {BRANCH} | commit: {local_git('rev-parse', '--short', 'HEAD')}")

if IN_COLAB:
    ensure_checkout()
else:
    print(f"Using local checkout: {REPO_DIR}")
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(REPO_DIR / "requirements/llada-colab.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e",
                str(REPO_DIR)], check=True)
import site
for package_dir in [*site.getsitepackages(), site.getusersitepackages()]:
    site.addsitedir(package_dir)

## 2. Persistent cache and GPU

The default Drive directory matches the MEDAL demo. Existing model snapshots are reused; otherwise the pinned Hub checkpoint uses the Drive cache. Change `DRIVE_ROOT` if your files are elsewhere.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/CSCI544-DLM") if IN_COLAB else REPO_DIR / ".cache/medal"
os.environ["HF_HUB_CACHE"] = str(DRIVE_ROOT / "cache/hub")
os.environ["HF_DATASETS_CACHE"] = str(DRIVE_ROOT / "cache/datasets")
DEMONSTRATIONS_PATH = DRIVE_ROOT / "manifests/gsm8k_decomposition_train.json"
MANIFEST_PATH = DRIVE_ROOT / "manifests/gsm8k_test.json"
OUTPUT_ROOT = DRIVE_ROOT / "results/ab_medal"

required = ["src/dlm_project/decoding/ab_medal.py", "scripts/run_ab_medal_single.py",
            "scripts/compare_medal_ab_medal.py"]
missing = [name for name in required if not (REPO_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"AB-MCTS files are missing from this checkout: {missing}. Push/copy them first.")

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before loading LLaDA.")
DTYPE = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"
print("GPU:", torch.cuda.get_device_name(0), "| dtype:", DTYPE)
print("Results:", OUTPUT_ROOT)

## 3. Search settings and model — load once

Leave `MAX_NFE = None` for the full-target check. Setting a cap limits initialization NFEs, including the root; a search that stops short of three candidates remains incomplete and is not decoded. Reducing the number of questions changes total evaluation work, not the search depth or candidate target for each question.

In [ ]:
import json
from dlm_project.decoding.ab_medal import ABMEDALConfig
from dlm_project.decoding.medal_demo import load_demo_model, prepare_demo_prompt
from dlm_project.experiments.artifacts import save_json
from dlm_project.experiments.paths import create_run_directory
from dlm_project.models.profiles import get_profile
from dlm_project.tasks.gsm8k.benchmark import example_seed
from dlm_project.tasks.gsm8k.benchmark_dataset import get_manifest
from scripts.run_ab_medal_single import run_attempt

EXAMPLE_INDEX, SEED = 0, 1
GEN_LENGTH, TEMPERATURE = 256, 1.0
MAX_NFE = None
SETTINGS = ABMEDALConfig(
    init_tokens=20, num_candidates=3, top_k_tokens=3, top_k_actions=5,
    max_model_calls=MAX_NFE, search_policy="gaussian",
    prior_m=0.0, prior_kappa=1.0, prior_nu=1.0, prior_tau2=0.1,
)

model_name = get_profile("llada")["repo_id"].split("/")[-1]
stored_models = [Path("/content/models") / model_name, DRIVE_ROOT / "models" / model_name]
MODEL_PATH = next((str(path) for path in stored_models if (path / "config.json").is_file()),
                  get_profile("llada")["repo_id"])
print("Model source:", MODEL_PATH)

In [ ]:
llada = load_demo_model(model_path=MODEL_PATH, device="cuda", dtype=DTYPE,
                        local_files_only=Path(MODEL_PATH).is_dir())

## 4. Full-target check: one pinned GSM8K test question

The test manifest and the two training demonstrations share a pinned dataset revision. The reference answer is used only to score the final decoded response. Full search events are printed and flushed to `trace.jsonl`; all reports and diagrams use cached search data.

In [ ]:
manifest = get_manifest(MANIFEST_PATH, cache_dir=os.environ["HF_DATASETS_CACHE"])
example = manifest["examples"][EXAMPLE_INDEX]
prompt, prompt_ids, prompt_meta = prepare_demo_prompt(
    llada, example["question"], DEMONSTRATIONS_PATH,
    revision=manifest["dataset"]["revision"], cache_dir=os.environ["HF_DATASETS_CACHE"],
)
directory, _ = create_run_directory({"output_dir": str(OUTPUT_ROOT / "single")})
save_json(directory / "prompt.json", {"text": prompt, "metadata": prompt_meta})
save_json(directory / "dataset.json", manifest["dataset"])
print("Question:", example["question"], "\nArtifacts:", directory)

report = run_attempt(
    llada, prompt_ids, SETTINGS, directory,
    seed=example_seed(SEED, example["row_id"]), gen_length=GEN_LENGTH,
    temperature=TEMPERATURE, reference=example["reference_solution"], metadata=example,
    prompt_seconds=prompt_meta["prompt_wall_seconds"], setup_seconds=prompt_meta["setup_wall_seconds"],
    print_history=True, plot_max_nodes=120, plot_max_depth=20,
)

In [ ]:
from IPython.display import Image, display

print(json.dumps({key: report[key] for key in (
    "status", "target_reached", "full_target_dlm_gate", "initialization_model_calls",
    "initialization_nfe", "decoding_nfe", "total_nfe", "accuracy", "response", "error",
)}, indent=2))
print("Evaluated depth:", report["structure"]["mean_evaluated_depth"],
      "(mean),", report["structure"]["max_evaluated_depth"], "(max)")
print("Candidate diversity:", report["structure"]["candidate_diversity"])
print("Full-target DLM gate:", "OBSERVED" if report["full_target_dlm_gate"] else "NOT SATISFIED")
display(Image(filename=str(directory / "ab_medal_tree.png")))

## 5. Optional five-example AB-MCTS pilot

Enable only after the genuine depth-20 / three-candidate check succeeds. The pilot reuses the loaded model and current settings on the first five pinned GSM8K test questions. Each question uses the repository's example-seed convention and has its own saved search artifacts.

The summary reports target coverage and conditional accuracy. Incomplete searches have unavailable correctness. Completed records are flushed after each question, and a normal notebook interruption retains a partial summary. This is an AB-MCTS pilot; paired MEDAL comparisons can be run separately with `scripts/compare_medal_ab_medal.py`.

In [ ]:
RUN_PILOT = False
if RUN_PILOT:
    if not report["full_target_dlm_gate"]:
        raise RuntimeError("Run and satisfy the full-target Gaussian DLM check first.")
    pilot_dir, _ = create_run_directory({"output_dir": str(OUTPUT_ROOT / "pilot")})
    rows, status, error = [], "running", None
    try:
        with (pilot_dir / "attempts.jsonl").open("w", encoding="utf-8") as output:
            for item in manifest["examples"][:5]:
                prompt, ids, meta = prepare_demo_prompt(
                    llada, item["question"], DEMONSTRATIONS_PATH,
                    revision=manifest["dataset"]["revision"], cache_dir=os.environ["HF_DATASETS_CACHE"],
                )
                attempt_dir = pilot_dir / f"example_{item['row_id']}"
                attempt_dir.mkdir()
                save_json(attempt_dir / "prompt.json", {"text": prompt, "metadata": meta})
                row = run_attempt(
                    llada, ids, SETTINGS, attempt_dir,
                    seed=example_seed(SEED, item["row_id"]), gen_length=GEN_LENGTH,
                    temperature=TEMPERATURE, reference=item["reference_solution"], metadata=item,
                    prompt_seconds=meta["prompt_wall_seconds"], setup_seconds=meta["setup_wall_seconds"],
                    print_history=False, plot_max_nodes=120, plot_max_depth=20,
                )
                rows.append(row)
                output.write(json.dumps(row, allow_nan=False) + "\n")
                output.flush()
                if row["status"] == "interrupted":
                    raise KeyboardInterrupt(row["error"])
                if row["status"] == "error":
                    raise RuntimeError(row["error"])
        status = "complete"
    except (Exception, KeyboardInterrupt) as exc:
        status = "interrupted" if isinstance(exc, KeyboardInterrupt) else "error"
        error = f"{type(exc).__name__}: {exc}"
    finally:
        scored = [row for row in rows if row["status"] == "complete" and row["accuracy"] is not None]
        summary = {
            "status": status, "error": error, "requested": 5, "processed": len(rows),
            "target_attained": sum(row["target_reached"] for row in rows),
            "target_coverage": sum(row["target_reached"] for row in rows) / 5,
            "scored": len(scored),
            "conditional_accuracy": sum(row["accuracy"] for row in scored) / len(scored) if scored else None,
            "total_nfe": sum(row["total_nfe"] for row in rows),
            "artifact_directory": str(pilot_dir),
        }
        save_json(pilot_dir / "summary.json", summary)
        print(json.dumps(summary, indent=2))